---
title: "DRG Splitting"

author: "Carlos Resurreccion"

date: "2025-02-27"

---


# !!! MAKE SURE YOU RESTART THE (JUPYTER) R KERNEL BEFORE PROCEEDING !!!


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in 
`~/drg-pipeline/data-cleaning/00a-parameters.r`


Change seldom touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`

In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


# Libraries


In [ ]:
source(here::here("data-cleaning", "00c-packages.r"))


# R Scripts


In [ ]:
source(here::here("data-cleaning", "00d-load-params-and-scripts.r"))


# Data Cleaning Proper


# Load Mapping Data


In [ ]:
source(here::here("data-cleaning", "00e-load-mapping.r"))


# Data Cleaning Function


In [ ]:
# from process_chunk (drg-cleaning)
# Define process_chunk
process_chunk <- function(
    chunk, yr_to_load = year, col_maps = column_mappings,
    known_vals = known_values, remap_master = col_remap_master,
    avail_cols = available_columns,
    looppart = loop_part) {
  fork_id <- Sys.getpid() # Get the process ID
  ##############################################################################
  # 1. Input/Year Standardization
  ##############################################################################
  # Rename Columns
  setnames(chunk,
    old = avail_cols[avail_cols %in% names(col_maps)],
    new = unlist(col_maps[avail_cols[avail_cols %in% names(col_maps)]])
  )

  # Safeguard: If id_year doesn't exist, set it to yr_to_load
  if (!"id_year" %in% names(chunk)) {
    set(chunk, j = "id_year", value = yr_to_load)
  }
  # Safeguard: If pat_bwt doesn't exist, set it to NA_real_
  if (!"pat_bwt" %in% names(chunk)) {
    set(chunk, j = "pat_bwt", value = NA_real_)
  }
  # Safeguard: If pat_bdate doesn't exist, set it to NA_Date_
  if (!"pat_bdate" %in% names(chunk)) {
    set(chunk, j = "pat_bdate", value = NA_Date_)
  }
  # Safeguard: If pat_ageday doesn't exist, set it to NA_real_
  if (!"pat_ageday" %in% names(chunk)) {
    set(chunk, j = "pat_ageday", value = NA_integer_)
  }

  return(chunk)
}


# Data Cleaning Loop

In [ ]:
# Data Cleaning Pipeline for DRG Processing
# This script processes large datasets in parts, applying
# parallel processing for efficiency.
# It reads, chunks, processes, and consolidates data before
# saving intermediate and final outputs.
abs_start_time <- as.character(Sys.time())
setkey(claims, id_series) # Do this ONCE before looping over chunks
for (loop_part in 1:split_parts) {
  start_time <- Sys.time() # Record start time for processing
  # Step 1: Read the appropriate file
  read_in_dt <- read_appropriate_file(loop_part)
  if (to_filter) {
    if ("PSEUDO_CLAIMSERIES" %in% names(read_in_dt)) {
      read_in_dt <- read_in_dt[, PSEUDO_CLAIMSERIES := trimws(as.character(PSEUDO_CLAIMSERIES))][claims, nomatch = 0, on = .(PSEUDO_CLAIMSERIES = id_series)]
    } else {
      read_in_dt <- read_in_dt[, CLAIM_SERIES_ID := trimws(as.character(CLAIM_SERIES_ID))][claims, nomatch = 0, on = .(CLAIM_SERIES_ID = id_series)]
    }
  }

  # Step 2: Split the data into chunks for parallel processing
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(read_in_dt, rep(1:nthreads,
    each = chunk_size,
    length.out = nrow(read_in_dt)
  ))

  # Step 3: Process chunks in parallel or sequentially
  cat(paste0("\rStart processing part  ", loop_part, " of ", split_parts))
  flush.console()

  parallel_results <-
    if (to_parallel) {
      mclapply(chunks, process_chunk, mc.cores = nthreads)
    } else if (!to_debug) {
      lapply(chunks, process_chunk)
    } else if (to_debug) {
      list(process_chunk(chunks[[1]]))
    } else {
      stop("Invalid parameters")
    }


  # Consolidate processed chunks
  summarized_dt <- rbindlist(parallel_results)

  # Step 4: Save processed data if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(filtered_chkpt_1_path, paste0(
        chkpt_1_prefix, year, suffix, "raw_master_",
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 5: Log processing time and update status
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(),
    start_time,
    units = "secs"
  ))
  print_status_update(loop_part, split_parts, processing_times, "clean")
  # Cleanup memory
  rm(read_in_dt, summarized_dt)
  invisible(gc())
}

# Step 6: Combine all processed parts into a master data table
master_dt_list <- mclapply(1:split_parts,
  function(split_part) {
    read_part <- readRDS(
      here(filtered_chkpt_1_path, paste0(
        chkpt_1_prefix, year, suffix, "raw_master_", "part_",
        sprintf("%02d", split_part), "_of_", split_parts, ".rds"
      ))
    )
    return(read_part)
  },
  mc.cores = nthreads
)

# Merge all parts into a single data table
master_dt <- rbindlist(master_dt_list, fill = TRUE)
rm(master_dt_list)
invisible(gc())

# Step 7: Save final processed data
if (to_write) {
  saveRDS(master_dt, here(
    filtered_chkpt_2_path, paste0(
      chkpt_2_prefix, year, suffix, "raw_master", ".rds"
    )
  ), compress = FALSE)
}


# BQ Upload

In [ ]:
# BQ upload
if (to_bq) {
  # Define BQ table name
  if (!to_sample) bq_table <- paste0("raw_claims_", year) else paste0("raw_claims_", year, suffix)

  # Attempt to delete the table if it exists
  tryCatch(
    bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table)),
    error = function(e) {
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.")
      } else {
        stop(e)
      }
    }
  )

  # Create the BQ table if it does not exist
  tryCatch(
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here(
        "data-cleaning/r_scripts_v2",
        "bq_schema_cleaning.json"
      ), simplifyDataFrame = FALSE)
    ),
    error = function(e) {
      if (grepl("already exists", e, ignore.case = TRUE)) {
        message("Table already exists. Skipping creation and upload.")
      } else {
        stop(e)
      }
    }
  )

  if (to_bq) {
    # Define chunk size for upload
    chunk_size <- 250000
    # Calculate number of chunks
    num_chunks <- ceiling(nrow(result) / chunk_size)

    for (i in seq_len(num_chunks)) {
      # Extract chunk
      chunk <- result[
        ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)),
      ]

      # Upload chunk to BQ
      bq_table_upload(
        bq_table(gcp_proj, bq_dataset, bq_table),
        values = chunk,
        write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
      )
    }
  }
}
